# 2차 — 블렌딩 방식 비교 (가중평균 vs Rank vs Isotonic 보정)

## 목적
지금까지는 가중평균 블렌딩만 썼는데(우리팀 0.6 + v5 0.4 → OOF 0.74071), 다른 블렌딩 방식이 더 나은지 확인.

## 누수 방지 설계 (중요)
- **Rank 블렌딩**: 점수를 백분위로 바꾸는 변환이라 y를 전혀 안 씀 → 누수 위험 자체가 없음
- **Isotonic 보정**: y를 쓰므로 **반드시 nested CV** — 매 fold마다 학습용 4-fold에만 보정기를 `fit`하고, held-out fold에는 `transform`만 적용 (절대 fit 안 함)
- **가중치 탐색**: 모델이 여러 개라 OOF 전체에 대해 그냥 최적화하면 과적합 위험 → 이것도 같은 5-fold 구조로 **nested 탐색** (학습 fold로 가중치 찾고 held-out fold로 평가)
- 즉, 모든 단계가 "평가용 fold는 그 fold의 가중치/보정 학습에 절대 쓰이지 않는다"는 원칙을 지킴

## 자동 탐색
`blend_cache/*_oof.npy`를 스캔해서 `{이름}_test.npy`가 있으면 같이 로드. 정확한 파일명을 모르니 동적으로 찾습니다 — 로드된 목록을 보고 빠진 모델이 있으면 알려주세요.

---
**저장 위치:** `experiment_history/2차/2차_blend_comparison.ipynb`


In [8]:
import json
import glob
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score


In [9]:
DATA_DIR = Path("../../data")
SHARED_DIR = Path("..")
BLEND_DIR = SHARED_DIR / "blend_cache"
SUBMIT_DIR = Path("../submit file")

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
TARGET_COL = "임신 성공 여부"

N_SPLITS = 5
SEED = 42


## 1. 타깃 로드 + CV 폴드 재구성 (전처리 불필요 — y만 필요)

In [10]:
y = pd.read_csv(TRAIN_PATH)[TARGET_COL].values.astype(int)
test_ids = pd.read_csv(TEST_PATH)["ID"]

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = list(skf.split(np.zeros(len(y)), y))
print(f"y 로드 완료: {len(y)}개, 양성비율={y.mean():.4f}")
print(f"{N_SPLITS}-fold 분할 완료 (seed={SEED})")


y 로드 완료: 256351개, 양성비율=0.2583
5-fold 분할 완료 (seed=42)


## 2. blend_cache 자동 스캔 + 개별 모델 OOF AUC 확인

In [11]:
# 두 가지 명명 규칙을 모두 인식:
#   - "{name}_oof.npy" / "{name}_test.npy"  (접미사, lambdarank/xgb_rankpairwise 등)
#   - "oof_{name}.npy" / "test_{name}.npy"  (접두사, catboost/xgboost 배깅 등)
# 예외(이름이 서로 다른 경우)는 MANUAL_TEST_MAP에 직접 매핑
MANUAL_TEST_MAP = {
    "oof_10seed_bagged.npy": "test_lgbm_bagged.npy",  # LightGBM 배깅만 oof/test 이름 stem이 다름
}

V5_DIR = SHARED_DIR / "팀원파일" / "김영혜"
V5_OOF_PATH = V5_DIR / "v5_ensemble_oof.npy"
V5_SUBMISSION_PATH = V5_DIR / "submission_v5_imputed.csv"

all_npy = sorted(glob.glob(str(BLEND_DIR / "*.npy")))
oof_candidates = [f for f in all_npy if "oof" in Path(f).stem.lower()]

models = {}
unmatched = []
for f in oof_candidates:
    fname = Path(f).name
    stem = Path(f).stem
    oof_arr = np.load(f)
    if len(oof_arr) != len(y):
        print(f"⚠ 스킵: {fname} (길이 불일치 oof={len(oof_arr)} vs y={len(y)})")
        continue

    if stem.startswith("oof_"):
        name = stem[len("oof_"):]
        default_test = BLEND_DIR / f"test_{name}.npy"
    elif stem.endswith("_oof"):
        name = stem[: -len("_oof")]
        default_test = BLEND_DIR / f"{name}_test.npy"
    else:
        name = stem
        default_test = None

    if fname in MANUAL_TEST_MAP:
        test_f = BLEND_DIR / MANUAL_TEST_MAP[fname]
    else:
        test_f = default_test

    entry = {"oof": oof_arr}
    if test_f is not None and test_f.exists():
        entry["test"] = np.load(test_f)
    else:
        unmatched.append(fname)
    models[name] = entry

# 팀원 v5 앙상블도 포함 (있으면) — test는 npy가 아니라 제출 csv에서 가져옴
if V5_OOF_PATH.exists():
    v5_oof = np.load(V5_OOF_PATH)
    if len(v5_oof) == len(y):
        entry = {"oof": v5_oof}
        if V5_SUBMISSION_PATH.exists():
            v5_sub = pd.read_csv(V5_SUBMISSION_PATH)
            entry["test"] = v5_sub["probability"].values
        models["v5_ensemble"] = entry
        print(f"v5_ensemble 포함됨 (test 예측 {'있음' if 'test' in entry else '없음'})")
    else:
        print(f"⚠ v5_ensemble_oof 길이 불일치라 제외 ({len(v5_oof)} vs {len(y)})")

model_names = list(models.keys())

# ★ 안전장치: 'y' 자체이거나 OOF AUC가 비정상적으로 1에 가까운 항목은 모델이 아니라
# 타깃(정답)이 잘못 끼어든 것일 가능성이 매우 높음 — 자동 제외 (그대로 두면 가중치 탐색이
# "정답 베끼기"로 수렴해서 모든 결과가 1.0으로 오염됨)
SUSPICIOUS_AUC_THRESHOLD = 0.995
removed = []
for n in list(model_names):
    if n == "y":
        removed.append((n, "이름이 'y' — 타깃 파일로 추정"))
        del models[n]
        continue
    auc_check = roc_auc_score(y, models[n]["oof"])
    if auc_check >= SUSPICIOUS_AUC_THRESHOLD:
        removed.append((n, f"OOF AUC={auc_check:.5f} — 비정상적으로 높음, 타깃 누출 의심"))
        del models[n]

if removed:
    print("⚠ 모델 목록에서 자동 제외됨 (타깃/누출 의심):")
    for n, reason in removed:
        print(f"   - {n}: {reason}")
    print()

model_names = list(models.keys())
print(f"발견된 모델: {model_names}")
if unmatched:
    print(f"⚠ test 짝을 못 찾은 OOF 파일 (OOF 비교에는 포함, 최종 제출에는 제외): {unmatched}")
print()
print(f"{'모델':<30} | {'OOF AUC':>8} | {'test 예측':>8}")
print("-" * 55)
for n in model_names:
    auc = roc_auc_score(y, models[n]["oof"])
    has_test = "있음" if "test" in models[n] else "없음(제외)"
    print(f"{n:<30} | {auc:>8.5f} | {has_test:>8}")

print()
print("모델간 OOF 상관관계:")
corr = pd.DataFrame({n: models[n]["oof"] for n in model_names}).corr()
print(corr.round(3))


v5_ensemble 포함됨 (test 예측 있음)
⚠ 모델 목록에서 자동 제외됨 (타깃/누출 의심):
   - y: 이름이 'y' — 타깃 파일로 추정

발견된 모델: ['lgbm_lambdarank', '10seed_bagged', 'age_split', 'catboost', 'catboost_bagged', 'lgbm', 'lgbm_robust_search', 'lgbm_tuned', 'tt', 'xgboost', 'xgboost_bagged', 'xgb_rankpairwise', 'v5_ensemble']
⚠ test 짝을 못 찾은 OOF 파일 (OOF 비교에는 포함, 최종 제출에는 제외): ['oof_age_split.npy', 'oof_catboost.npy', 'oof_lgbm.npy', 'oof_lgbm_robust_search.npy', 'oof_lgbm_tuned.npy', 'oof_tt.npy', 'oof_xgboost.npy', 'oof_y.npy']

모델                             |  OOF AUC |  test 예측
-------------------------------------------------------
lgbm_lambdarank                |  0.73748 |       있음
10seed_bagged                  |  0.74036 |       있음
age_split                      |  0.73922 |   없음(제외)
catboost                       |  0.73971 |   없음(제외)
catboost_bagged                |  0.74005 |       있음
lgbm                           |  0.73925 |   없음(제외)
lgbm_robust_search             |  0.74037 |   없음(제외)
lgbm_tuned              

## 3. 누수 없는 nested CV 가중치 탐색 함수

In [12]:
def softmax_weights(theta):
    e = np.exp(theta - np.max(theta))
    return e / e.sum()


def optimize_weights(feature_matrix, y_sub, n_models, n_restarts=3, seed=0):
    """학습 데이터(feature_matrix, y_sub)만으로 블렌딩 가중치 탐색. 여기엔 held-out fold가 절대 섞이지 않음."""
    rng = np.random.RandomState(seed)
    best_w, best_auc = None, -1.0
    inits = [np.zeros(n_models)] + [rng.normal(0, 1, n_models) for _ in range(n_restarts - 1)]
    for theta0 in inits:
        def neg_auc(theta):
            w = softmax_weights(theta)
            return -roc_auc_score(y_sub, feature_matrix @ w)
        res = minimize(neg_auc, theta0, method="Nelder-Mead",
                        options={"maxiter": 3000, "xatol": 1e-4, "fatol": 1e-6})
        w = softmax_weights(res.x)
        auc = roc_auc_score(y_sub, feature_matrix @ w)
        if auc > best_auc:
            best_auc, best_w = auc, w
    return best_w


def nested_cv_weighted_blend(feature_matrix, y, folds):
    """가중치만 nested CV로 탐색 (feature_matrix 자체는 이미 y와 무관하게 만들어졌다고 가정 — raw/rank용)"""
    n_models = feature_matrix.shape[1]
    blended_oof = np.zeros(len(y))
    fold_weights = []
    for tr_idx, te_idx in folds:
        w = optimize_weights(feature_matrix[tr_idx], y[tr_idx], n_models)
        blended_oof[te_idx] = feature_matrix[te_idx] @ w
        fold_weights.append(w)
    return blended_oof, np.array(fold_weights)


def nested_cv_isotonic_blend(oof_dict, y, folds):
    """보정기(isotonic)도 매 fold마다 학습 fold에만 fit, held-out fold는 transform만 적용"""
    names = list(oof_dict.keys())
    oof_matrix = np.column_stack([oof_dict[n] for n in names])
    n_models = len(names)
    blended_oof = np.zeros(len(y))
    fold_weights = []
    for tr_idx, te_idx in folds:
        train_mat, test_mat = oof_matrix[tr_idx], oof_matrix[te_idx]
        y_tr = y[tr_idx]
        cal_train = np.zeros_like(train_mat)
        cal_test = np.zeros_like(test_mat)
        for j in range(n_models):
            iso = IsotonicRegression(out_of_bounds="clip")
            iso.fit(train_mat[:, j], y_tr)          # 학습 fold에만 fit
            cal_train[:, j] = iso.transform(train_mat[:, j])
            cal_test[:, j] = iso.transform(test_mat[:, j])  # held-out엔 적용만
        w = optimize_weights(cal_train, y_tr, n_models)
        blended_oof[te_idx] = cal_test @ w
        fold_weights.append(w)
    return blended_oof, np.array(fold_weights)


## 4. 세 가지 블렌딩 방식 비교

In [13]:
raw_matrix = np.column_stack([models[n]["oof"] for n in model_names])

# rank 변환은 y를 안 써서 nested 필요 없음 (전역으로 한 번만 계산해도 누수 없음)
rank_matrix = np.column_stack([rankdata(models[n]["oof"]) / len(y) for n in model_names])

results = {}

oof_raw, w_raw = nested_cv_weighted_blend(raw_matrix, y, folds)
results["raw_weighted"] = {"oof": oof_raw, "auc": roc_auc_score(y, oof_raw), "weights": w_raw.mean(axis=0)}

oof_rank, w_rank = nested_cv_weighted_blend(rank_matrix, y, folds)
results["rank_weighted"] = {"oof": oof_rank, "auc": roc_auc_score(y, oof_rank), "weights": w_rank.mean(axis=0)}

oof_iso, w_iso = nested_cv_isotonic_blend({n: models[n]["oof"] for n in model_names}, y, folds)
results["isotonic_weighted"] = {"oof": oof_iso, "auc": roc_auc_score(y, oof_iso), "weights": w_iso.mean(axis=0)}

print(f"{'방식':<20} | {'Nested CV 블렌딩 OOF AUC':>22}")
print("-" * 48)
for k, v in results.items():
    print(f"{k:<20} | {v['auc']:>22.5f}")

print()
print("비교 기준:")
print("  멀티시드 배깅 OOF: 0.74037")
print("  팀 블렌딩(가중평균, 2개 모델) OOF: 0.74071")
print()
for k, v in results.items():
    print(f"[{k}] 평균 가중치:")
    for n, w in zip(model_names, v["weights"]):
        print(f"   {n}: {w:.4f}")


방식                   |  Nested CV 블렌딩 OOF AUC
------------------------------------------------
raw_weighted         |                0.74063
rank_weighted        |                0.74065
isotonic_weighted    |                0.74060

비교 기준:
  멀티시드 배깅 OOF: 0.74037
  팀 블렌딩(가중평균, 2개 모델) OOF: 0.74071

[raw_weighted] 평균 가중치:
   lgbm_lambdarank: 0.0002
   10seed_bagged: 0.1198
   age_split: 0.0072
   catboost: 0.0210
   catboost_bagged: 0.0164
   lgbm: 0.0032
   lgbm_robust_search: 0.1597
   lgbm_tuned: 0.0054
   tt: 0.0764
   xgboost: 0.0124
   xgboost_bagged: 0.1790
   xgb_rankpairwise: 0.0008
   v5_ensemble: 0.3986
[rank_weighted] 평균 가중치:
   lgbm_lambdarank: 0.0002
   10seed_bagged: 0.1853
   age_split: 0.0043
   catboost: 0.0247
   catboost_bagged: 0.0371
   lgbm: 0.0016
   lgbm_robust_search: 0.0373
   lgbm_tuned: 0.0116
   tt: 0.0685
   xgboost: 0.0444
   xgboost_bagged: 0.1346
   xgb_rankpairwise: 0.0008
   v5_ensemble: 0.4496
[isotonic_weighted] 평균 가중치:
   lgbm_lambdarank: 0.0010
   

## 5. 최적 방식으로 최종 test 예측 생성 + 저장

In [14]:
best_method = max(results, key=lambda k: results[k]["auc"])
print(f"채택: {best_method} (nested CV 블렌딩 OOF AUC = {results[best_method]['auc']:.5f})")

models_with_test = [n for n in model_names if "test" in models[n]]
missing = set(model_names) - set(models_with_test)
if missing:
    print(f"⚠ test 예측 없어서 최종 제출 블렌딩에서 제외됨: {missing}")

oof_sub = np.column_stack([models[n]["oof"] for n in models_with_test])
test_sub = np.column_stack([models[n]["test"] for n in models_with_test])
n_sub = len(models_with_test)

if best_method == "raw_weighted":
    w_final = optimize_weights(oof_sub, y, n_sub)
    test_pred_final = test_sub @ w_final

elif best_method == "rank_weighted":
    rank_oof_sub = np.column_stack([rankdata(models[n]["oof"]) / len(y) for n in models_with_test])
    w_final = optimize_weights(rank_oof_sub, y, n_sub)
    rank_test_sub = np.column_stack([rankdata(models[n]["test"]) / len(models[n]["test"]) for n in models_with_test])
    test_pred_final = rank_test_sub @ w_final

else:  # isotonic_weighted
    cal_oof_sub = np.zeros_like(oof_sub)
    cal_test_sub = np.zeros_like(test_sub)
    for j, n in enumerate(models_with_test):
        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(models[n]["oof"], y)  # 이제는 평가가 끝났으니 전체 train에 fit (test엔 라벨이 없어 누수 불가능)
        cal_oof_sub[:, j] = iso.transform(models[n]["oof"])
        cal_test_sub[:, j] = iso.transform(models[n]["test"])
    w_final = optimize_weights(cal_oof_sub, y, n_sub)
    test_pred_final = cal_test_sub @ w_final

print(f"최종 가중치: {dict(zip(models_with_test, np.round(w_final, 4)))}")

BLEND_DIR.mkdir(exist_ok=True)
np.save(BLEND_DIR / f"blend_{best_method}_test.npy", test_pred_final)

best_oof_auc = results[best_method]["auc"]
SUBMIT_DIR.mkdir(exist_ok=True)
submission = pd.DataFrame({"ID": test_ids, "probability": test_pred_final})
out_path = SUBMIT_DIR / f"2차_blend_{best_method}_local{best_oof_auc:.5f}.csv"
submission.to_csv(out_path, index=False)
print(f"비제출용 결과 저장: {out_path}")


채택: rank_weighted (nested CV 블렌딩 OOF AUC = 0.74065)
⚠ test 예측 없어서 최종 제출 블렌딩에서 제외됨: {'age_split', 'lgbm', 'tt', 'lgbm_robust_search', 'catboost', 'xgboost', 'lgbm_tuned'}
최종 가중치: {'lgbm_lambdarank': np.float64(0.0), '10seed_bagged': np.float64(0.2481), 'catboost_bagged': np.float64(0.1094), 'xgboost_bagged': np.float64(0.1961), 'xgb_rankpairwise': np.float64(0.0), 'v5_ensemble': np.float64(0.4463)}
비제출용 결과 저장: ../submit file/2차_blend_rank_weighted_local0.74065.csv


## 6. 해석

- 세 방식 중 nested CV 기준 가장 높은 OOF AUC가 0.74071(기존 팀 블렌딩)을 넘으면, 블렌딩 방식 자체를
  바꿔서 얻은 개선 — 제출 후보로 검토할 가치 있음
- 못 넘으면, 2개 모델 단순 가중평균이 이미 충분히 좋았다는 뜻 (모델간 상관관계가 0.9+로 높다면, 어떤
  블렌딩 방식을 쓰든 비슷한 천장에 도달하는 게 자연스러움 — 위 상관관계 표 참고)
- `blend_cache/blend_{방식}_test.npy`는 다음 블렌딩 실험에서 추가 후보로 재사용 가능
